# Subsetting Qiu et al. to blood related cells only

In [1]:
suppressPackageStartupMessages(library(Seurat))
suppressPackageStartupMessages(library(SingleCellExperiment))
suppressPackageStartupMessages(library(data.table))
suppressPackageStartupMessages(library(dplyr))

Warning message:
“package ‘Seurat’ was built under R version 4.1.2”


In [2]:
files = list.files()[grep('seurat',list.files())]

In [3]:
meta_full = lapply(files, function(x){
    tmp = readRDS(file.path(getwd(), x))
    return(as.data.table(tmp@meta.data, keep.rownames=T) %>% setnames('rn', 'cell'))
}) %>%
    rbindlist(fill=TRUE) %>%
    .[,dataset:='qiu']

fwrite(meta_full, file.path(getwd(), 'meta_full.txt.gz'))

In [4]:
ct_keep = c('Endothelium',
            'Primitive erythroid cells', 
            'Definitive erythroid cells',
            'Megakaryocytes', 
            'Hematoendothelial progenitors',
            'Blood progenitors')

In [5]:
nrow(meta_full[cell_type %in% ct_keep])

[1] 73690

In [6]:
fwrite(meta_full[cell_type %in% ct_keep], file.path(getwd(), 'meta_blood.txt.gz'))

In [7]:
sce_list = lapply(files, function(x){ 
    tmp = readRDS(file.path(getwd(), x))
    tmp = tmp[,tmp@meta.data$cell_type%in%ct_keep]
    sce = as.SingleCellExperiment(tmp)
    colData(sce) = meta_full[match(colnames(sce), meta_full$cell)] %>% 
                            as.data.frame %>% 
                            tibble::column_to_rownames("cell") %>%
                            .[colnames(sce),] %>% DataFrame()
    return(sce)
})
big_sce = do.call('cbind', sce_list)
saveRDS(big_sce, file.path(getwd(), 'qiu_blood_sce.rds'))

In [8]:
big_sce

class: SingleCellExperiment 
dim: 24552 73690 
metadata(0):
assays(2): counts logcounts
rownames(24552): ENSMUSG00000051951 ENSMUSG00000102343 ...
  ENSMUSG00000064368 ENSMUSG00000064370
rowData names(0):
colnames(73690): sci3-me-001.CTACGGCATGCTAACTTGC
  sci3-me-002.AGATTGGTTTTCTAATAGTA ... sci3-me-758.AGTTGCGCTAATTAAGACT
  sci3-me-760.TTCGCGGATAATTAAGACT
colData names(10): orig.ident nCount_RNA ... somite_stage dataset
reducedDimNames(0):
mainExpName: RNA
altExpNames(0):